In [ ]:
import pandas as pd
import numpy as np
import os
os.environ['KMP_DUPLICATE_LIB_OK'] = 'True'
export_dir = os.getcwd()
from pathlib import Path
import pickle
from collections import defaultdict
import time
import torch
import torch.nn as nn
import copy
import torch.nn.functional as F
import optuna
import logging
import matplotlib.pyplot as plt
import ipynb
import importlib
import sys
from pathlib import Path
from collections import Counter
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.model_selection import train_test_split 
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
import numpy as np
import xgboost as xgb
from sklearn.metrics import r2_score
from sklearn.metrics import mean_squared_error


## Dataset and the Model

In [ ]:
data_name = "ML1M" ### Can be ML1M, LASTFM
recommender_name = "MLP" ## Can be MLP, VAE, MLP_model, GMF_model, NCF
DP_DIR = Path("data/", data_name) 
export_dir = Path(os.getcwd())
files_path = Path(export_dir, DP_DIR)
checkpoints_path = Path(export_dir, "checkpoints")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
output_type_dict = {
    "VAE":"multiple",
    "MLP":"single"
}

recommender_path_dict = {
    ("ML1M","VAE"): Path(checkpoints_path, "VAE_ML1M_1_15_0.0008_64.pt"),
    ("ML1M","MLP"):Path(checkpoints_path, "MLP_ML1M_0.0001_512_0_13.pt"),
    ("LASTFM","MLP"): Path(checkpoints_path, "MLP_lastfm_0.0001_512_0_16.pt"),
    ("LASTFM","VAE"): Path(checkpoints_path, "VAE_new_lastfm_0_21_0.0_256.pt"),

    ("Yahoo","VAE"): Path(checkpoints_path, "VAE_Yahoo_0.0001_128_13.pt"),
    ("Yahoo","MLP"):Path(checkpoints_path, "MLP2_Yahoo_0.0083_128_1.pt"),

    ("Pinterest","VAE"): Path(checkpoints_path, "VAE_Pinterest_0.0002_32_12.pt"),
    ("Pinterest","MLP"):Path(checkpoints_path, "MLP_Pinterest_0.0062_512_21_0.pt")
    
}

hidden_dim_dict = {
    ("ML1M","VAE"): None,
    ("ML1M","MLP"): 128,

    ("Yahoo","VAE"): None,
    ("Yahoo","MLP"):32,
    
    ("Pinterest","VAE"): None,
    ("Pinterest","MLP"):512,
        ("LASTFM", "MLP"): 128,
                ("LASTFM", "VAE"): 128


}

In [ ]:
output_type = output_type_dict[recommender_name] ### Can be single, multiple
hidden_dim = hidden_dim_dict[(data_name,recommender_name)]
recommender_path = recommender_path_dict[(data_name,recommender_name)]

## Training and Test Datasets

In [ ]:
## Load the data
train_data = pd.read_csv(Path(files_path,f'Train_Data_{data_name}_New.csv'), index_col=0)

test_data = pd.read_csv(Path(files_path,f'Test_Data_{data_name}_New.csv'), index_col=0)

#static_test_data = pd.read_csv(Path(files_path,f'Static_Test_Data_{data_name}_New.csv'), index_col=0)

## Load the items data and the popularity dictionary 
# items_data: meta features like genre

items_data = pd.read_csv(Path(files_path,f'Items_Data_{data_name}_New.csv'), index_col=0)
with open(Path(files_path,f'pop_dict_{data_name}.pkl'), 'rb') as f:
    pop_dict = pickle.load(f)

    
train_array = train_data.to_numpy()
test_array = test_data.to_numpy()
num_items=len(train_data.iloc[:,:-1].columns)
num_users=np.concatenate([train_array,test_array],axis=0).shape[0]

items_array = np.eye(num_items)
all_items_tensor = torch.Tensor(items_array).to(device)
all_array=np.concatenate((train_array, test_array))

## df for training and tes datasets without the users index
all_df_data=pd.concat([train_data,test_data], axis=0 )
#all_df_data.columns=all_df_data.columns.astype(int)

In [ ]:
all_df_data.drop(columns=["user_ids"], inplace=True)

In [ ]:
num_items=len(all_df_data.columns)
num_users=len(all_df_data)
num_interactions=all_df_data.sum().sum()

In [ ]:
sparsity = 1 - (num_interactions / (num_users * num_items))
print(f"sparsity: {sparsity}")

In [ ]:
#df_final=df_final.rename(columns={"user_ids":"user_id:token", "item_id": "item_id:token", "rating": "rating:float" })

In [ ]:
#df_final.to_csv('my_dataset_ML1M.inter', sep='\t', index=False)

In [ ]:
items_data.rename(columns={'track_id':'item_id'}, inplace=True)

In [ ]:
df_inter=pd.read_csv('/home/mvarasteh/ProvidersExplantion/data/LASTFM/user_data_playcount.csv', index_col=0)

In [ ]:
from sklearn.preprocessing import LabelEncoder

Artist_encoder = LabelEncoder()
items_data['artist_encoded'] = Artist_encoder.fit_transform(items_data['artist'])

In [ ]:
item2artist=items_data.set_index('item_id')['artist_encoded'].to_dict()

In [ ]:
pop_artist=dict(items_data['artist_encoded'].value_counts()/items_data['artist_encoded'].value_counts().max())

In [ ]:
all_list=[]
for k, v in pop_dict.items():
    all_list.append((pop_artist[item2artist[k]], v))

In [ ]:
with open('/home/mvarasteh/ProvidersExplantion/data/LASTFM/playcounts.pkl', 'rb') as f:
    playcounts = pickle.load(f)

In [ ]:
pop_array = np.zeros(len(pop_dict))
for key, value in pop_dict.items():
    pop_array[key] = value

## Importing recommendations

In [ ]:
from ipynb.fs.defs.recommenders_architecture import *
importlib.reload(ipynb.fs.defs.recommenders_architecture)
from ipynb.fs.defs.recommenders_architecture import *

## Importing help functions

In [ ]:
from ipynb.fs.defs.help_functions import *
importlib.reload(ipynb.fs.defs.help_functions)
from ipynb.fs.defs.help_functions import *

In [ ]:
kw_dict = {'device':device,
          'num_items': num_items,
          'pop_array':pop_array,
          'all_items_tensor':all_items_tensor,
          #'static_test_data':static_test_data,
          'items_array':items_array,
          'output_type':output_type,
          'recommender_name':recommender_name

        }

In [ ]:
VAE_config = { "enc_dims": [256, 64], "dropout": 0.5, "anneal_cap": 0.2, "total_anneal_steps": 200000 }


## Laoding recommender

In [ ]:
def load_recommender():
    if recommender_name=='MLP':
        recommender = MLP(hidden_dim, **kw_dict)
    elif recommender_name=='VAE':
        recommender = VAE(VAE_config, **kw_dict)

    recommender_checkpoint = torch.load(Path(checkpoints_path, recommender_path), map_location=torch.device('cpu'))
    recommender.load_state_dict(recommender_checkpoint)
    recommender.eval()
    for param in recommender.parameters():
        param.requires_grad= False
    return recommender
recommender = load_recommender()

## Genre of items

In [ ]:
## Converting each genre to a multihot vector 
ID2genre_hot=pd.concat([items_data["item_id"], items_data['genre'].str.get_dummies(sep='|')], axis=1)

In [ ]:
def genre(input_array, item2genre, users, Targ_genre):
    
    '''
    Counting the occurences of each genre in users profiles
    input_array: all users interactions
    item2genre: itmes to genre dataframe(multihot)
    users: all the users 
    Targ_genre: genres of the target item
    '''
    A=input_array[:,:-1]

    if not users:
        return 0
    
    else:
         
         z=sum( item2genre.set_index('item_id').loc[np.nonzero(A[uid])[0]].sum(axis=0) for uid in users)/len(users)
         
         return z[Targ_genre].mean()


## Target Items Time (Normalization)

In [ ]:
#items_data['year'] = items_data['name'].str.extract(r'\((\d{4})\)', expand=False).astype('int')
ID_Year=items_data.set_index('item_id')['year']
## Normalization
#max_val=ID_Year.max()
#min_val=ID_Year.min()
#ID_Year=(ID_Year-min_val)/(max_val-min_val)

## Recency of users profile

In [ ]:
def recency_profiles(input_array, users, ID_Year):
    
    if not users:

        return 0  
    
    else: 
        avg_time_all=[]
        for u in users:
            user_profile=input_array[u,:-1]
            idx = np.nonzero(user_profile)[0]  # indices of interacted items

            avg_time=ID_Year[idx].mean()
            avg_time_all.append(avg_time)
        return np.mean(avg_time_all)


## Generating recomemnadtions

In [ ]:
def recommendations_generation(input_array):
    topk_itms = {}

    for i in range(input_array.shape[0]):
        user_index = input_array[i][-1]
        user_tensor = torch.Tensor(input_array[i][:-1]).to(device)
        
        topk_itms[user_index] = get_user_recommended_item(user_tensor, recommender, **kw_dict).cpu().numpy()[0:10]
    return topk_itms
topk_itms=recommendations_generation(all_array)

In [ ]:
sorted_pop=sorted(pop_dict.items(), key=lambda x:x[1], reverse=True)[:100]
Topk_pop, _=list(zip(*sorted_pop))

In [ ]:
recomms_list=np.concatenate(list(topk_itms.values()))
#mutul_itms=recomms_set.intersection(set(Topk_pop))


## Convert items recommendation dict to an array

In [ ]:
recommendation_matrix = np.zeros((num_users, num_items), dtype=int)
for user, items in topk_itms.items():
    recommendation_matrix[user, items] = 1  

## Item2users

In [ ]:
## list of users for each item in recommendations
item2users=pd.DataFrame(list(topk_itms.items()), columns=['user_id', 'items_id']).explode('items_id').groupby('items_id')['user_id'].apply(list).to_dict()

## Exposure of all items

In [ ]:
all_values=np.concatenate(list(topk_itms.values()))
items_indx=np.unique(all_values)
items_indx=np.arange( num_items)
count=Counter(all_values)
#count_all={item: count.get(item, 0) for item in items_indx}

In [ ]:
exp_df=pd.DataFrame({"ids":count.keys(), "exp":count.values()})


In [ ]:
exp_df.to_csv(f"exp_df_{recommender_name}_{data_name}.csv")

## Users size

In [ ]:
users_size=(all_df_data. sum(axis=1).to_dict())

## Creating profile popularity feature

In [ ]:
def profile_popularity (input_array, users):  
      
      '''
      computing average profiles populairty for users that recived the target item as a reocommendation 
      input_array: alll the interactions for each user and item
      users: all the users that recived the target item as a reocmmendation
      '''
      if not users:
        return 0
      
      else:
        All_pop = [] ## stores all pop scores for each users
    
        for u_id in users:
            
            ## profile for users
            user_profile=input_array[u_id,0:-1]
            ## get popularity of items in user profile
            mu=sum(pop_dict[int(item)] for item in np.nonzero(user_profile)[0]) / len(np.nonzero(user_profile)[0])
            All_pop.append(mu)

        return np.mean(All_pop)


In [ ]:
def profile_playcount (input_array, users):  
      
      '''
      computing average profiles populairty for users that recived the target item as a reocommendation 
      input_array: alll the interactions for each user and item
      users: all the users that recived the target item as a reocmmendation
      '''
      if not users:
        return 0
      
      else:
        All_pop = [] ## stores all pop scores for each users
    
        for u_id in users:
            
            ## profile for users
            user_profile=input_array[u_id,0:-1]
            ## get popularity of items in user profile
            mu=sum(playcounts[int(item)] for item in np.nonzero(user_profile)[0]) / len(np.nonzero(user_profile)[0])
            All_pop.append(mu)

        return np.mean(All_pop)

In [ ]:


def Artist_profilr_pop (input_array, users):  
      
      '''
      computing average profiles populairty for users that recived the target item as a reocommendation 
      input_array: alll the interactions for each user and item
      users: all the users that recived the target item as a reocmmendation
      '''
      if not users:
        return 0
      
      else:
        All_pop = [] ## stores all pop scores for each users
    
        for u_id in users:
            
            ## profile for users
            user_profile=input_array[u_id,0:-1]
            ## get popularity of items in user profile
            mu=sum(pop_artist[item2artist[item]] for item in np.nonzero(user_profile)[0]) / len(np.nonzero(user_profile)[0])
            All_pop.append(mu)

        return np.mean(All_pop)

## Profile size feature

In [ ]:

def profile_size(input_array, users, users_size):

    
    """
    input_array: np.array of shape [num_users, num_items+1]
                 Last column is user_id
    users: list of user_ids who received the target item as recommendation
    
    Returns:
        Average normalized profile size for the users
    """
   
    if not users:

        return 0
    
    else: 
    
        
        return sum(users_size[uid] for uid in users)/len(users)


## Finding similiar users for each user_id

## Based on embeddings

In [ ]:
users_embeddings=recommender.users_fc.weight.cpu().numpy()
users_embeddings=(all_array[:,:-1] @ (users_embeddings.T))
#cosine_sim_users=cosine_similarity(users_embeddings)
cosine_sim_users = np.corrcoef(users_embeddings)

mask=np.ones_like(cosine_sim_users)
np.fill_diagonal(mask,0)

similar_users_all = np.argsort(cosine_sim_users * mask)[:,::-1]

users_id=np.arange(similar_users_all.shape[0])
id_sim_users=similar_users_all[:,:10]
sim_users=dict(zip(users_id, id_sim_users))
df_sim_users=pd.DataFrame(sim_users.items(), index=None, columns=['user_id', 'similar_users'])

## Based on profiles

In [ ]:

#cosine_sim_users=cosine_similarity(all_array[:,:-1])
cosine_sim_users=np.corrcoef(all_array[:,:-1])
mask=np.ones_like(cosine_sim_users)
np.fill_diagonal(mask,0)

similar_users_all = np.argsort(cosine_sim_users * mask)[:,::-1]

users_id=np.arange(similar_users_all.shape[0])
id_sim_users=similar_users_all[:,:10]
sim_users=dict(zip(users_id, id_sim_users))
df_sim_users=pd.DataFrame(sim_users.items(), index=None, columns=['user_id', 'similar_users'])

In [ ]:
mask=np.ones_like(cosine_sim_users)
np.fill_diagonal(mask, 0)
mask_score=mask*cosine_sim_users
r,c=np.where(mask_score>0.7)

from collections import defaultdict

sim_users_dict = defaultdict(list)

# Use a standard loop to execute the appends
for r_idx, c_idx in zip(r, c):
    sim_users_dict[r_idx].append(c_idx)
sim_users_size={k: len(v) for k, v in sim_users_dict.items()}
infl_users,_=zip(*sorted(sim_users_size.items(), key=lambda x:x[1], reverse=True)[0:10])

## Counts how often the target item shows up in the  k neighbors profile

In [ ]:

def Targ_in_neighbs_profile (input_array, similar_users, users, targ_item, drop_last_col=True):
    """
    Count, for each user t, how many of t's similar users have targ_item (observed/consumed).
    input_array: 2D array [n_users, n_items( + maybe 1 label col)]
    similar_users: dict[int -> list[int]] of similar-user indices
    targ_item: int (column index in the **effective** item matrix)
    k: optional cap on number of similar users
    drop_last_col: set True if the last column is a label you want to ignore
    """
    if not users:
        return 0
    A = input_array[:, :-1] if drop_last_col else input_array
    # Boolean vector: does user u have targ_item?
    Target_has_item = A[:, targ_item] != 0

    dict_sim_users = dict(similar_users.iloc[users].values)
    
    counts = {}
    
    for t, S in dict_sim_users.items():
        #counts[t] = int(Target_has_item[np.asarray(S, dtype=int)].sum()) - int(Comp_has_item[np.asarray(S, dtype=int)].sum())
        counts[t] = int(Target_has_item[np.asarray(S, dtype=int)].sum()) 

    #for t, S in dict_sim_users.items():
     #   counts[t] = df_inter[(df_inter['track_id']==targ_item) & (df_inter['user_id'].isin(S))]['playcount'].sum()
    #min_val = min(counts.values())
    #max_val = max(counts.values())
    #range_val = (max_val - min_val)

    #Targ_in_neighb_prof = {k: (v - min_val) / range_val for k, v in counts.items()}
    
    #item2users_neighbprof=sum(counts[u] for u in item2users.get(targ_item,[0]) )/len (item2users.get(targ_item,[0]))
    item2users_neighbprof = sum(counts.values()) / len(counts)

    
    return item2users_neighbprof


## Counts how often the target item shows up in neighbors top-k recommenddadtions

In [ ]:

def Targ_in_neighbs_recomm(input_array, similar_users, users, targ_item, drop_last_col=False):

    """
    Count, for each item i, how many of i's similar users have targ_item (observed/consumed).
    input_array: 2D array [n_users, n_items( + maybe 1 label col)]
    similar_users: dict[int -> list[int]] of similar-user indices
    recommendations: dict[int -> list[int]] of recommended items for each user
    targ_item and Comp_item: int (column index in the **effective** item matrix)
    k: optional cap on number of similar users
    drop_last_col: set True if the last column is a label you want to ignore
    """
    if not users:
        return 0

    A = input_array[:, :-1] if drop_last_col else input_array
    # Boolean vector: does user u have targ_item?
    Target_has_item = A[:, targ_item] != 0


    dict_sim_users = dict(similar_users.iloc[users].values)
    counts = {}
    
    for t, S in dict_sim_users.items():
        #counts[t] = int(Target_has_item[np.asarray(S, dtype=int)].sum()) - int(Comp_has_item[np.asarray(S, dtype=int)].sum())
        counts[t] = int(Target_has_item[np.asarray(S, dtype=int)].sum()) 

    
    #min_val = min(counts.values())
    #max_val = max(counts.values())
    #range_val = (max_val - min_val)
    #print(f"range_val is {range_val}")
    #Targ_in_neighb_recomm = {k: (v - min_val) / range_val for k, v in counts.items()}


    #item2users_neighbrecomms=sum(counts[u] for u in item2users.get(targ_item,[0]) )/len (item2users.get(targ_item,[0]))
    
    item2users_neighbrecomms =  sum(counts.values()) / len(counts)


    return item2users_neighbrecomms




## Finding similiar items

In [ ]:
#cosine_sim_items=cosine_similarity(all_array[:,:-1].T)
cosine_sim_items=np.corrcoef(all_array[:,:-1].T)
#items_embeddings=np.array(recommender.items_fc.weight.detach())
#cosine_sim=cosine_similarity(items_embeddings.T)
mask=np.ones_like(cosine_sim_items)
np.fill_diagonal(mask,0)
similar_items_all = np.argsort(cosine_sim_items * mask)[:,::-1]
itms_id=np.arange(similar_items_all.shape[0])
id_sim_itms=similar_items_all[:,:10]
sim_itms=dict(zip(itms_id, id_sim_itms))

In [ ]:
mask=np.ones_like(cosine_sim_items)
np.fill_diagonal(mask, 0)
mask_score=mask*cosine_sim_items
r,c=np.where(mask_score>0.7)

In [ ]:
from collections import defaultdict

sim_items_dict = defaultdict(list)

# Use a standard loop to execute the appends
for r_idx, c_idx in zip(r, c):
    sim_items_dict[r_idx].append(c_idx)

In [ ]:
sim_users_size={k:len(v) for k, v in sim_items_dict.items()}

## count the number of similar items that are in the user profile


In [ ]:

def neighbors_item_profile(input_array, similar_items, users, targ_item, drop_last_col=True):
    """
    For each user u, count how many of u's items are in targ_item's similar-items list.
    input_array: 2D array [n_users, n_items (+ maybe 1 user-id col)]
                 Assumes item columns are binary (0/1).
    similar_items: dict[int -> list[int]]  (item -> similar item indices)
    targ_item: int (column index in the effective item matrix)
    drop_last_col: if True, treat the last column of input_array as a user-id (ignored for items)
    """
    A = input_array[:, :-1] if drop_last_col else input_array
    user_ids = input_array[:, -1].astype(int) if drop_last_col else np.arange(A.shape[0])

    # indices of items similar to targ_item
    sim_idx = np.array(similar_items.get(targ_item, []), dtype=int)

    if len(similar_items[targ_item]) == 0:
        return 0

    # Ensure binary (handles 0/1 or counts)
    Ab = (A != 0)

    # Count, per user, how many of their items intersect sim_idx (vectorized slice + sum)
    counts_vec = Ab[:, sim_idx].sum(axis=1)
    count={int(u): int(c) for u, c in zip(user_ids, counts_vec)}

    # Return dict[user_id] -> count
    #min_val = min(count.values())
    #max_val = max(count.values())
    #range_val = (max_val - min_val)

    #Neighb_items_in_prof = {k: (v - min_val) / range_val for k, v in count.items()}
    
    item2users_neighbritms = 0 if not users else sum(count[u] for u in users) / len(users)

    
    return item2users_neighbritms


In [ ]:
ID_energy=items_data.set_index('item_id')['energy']

def energy_profiles(input_array, users, ID_energy):
    
    if not users:

        return 0
    
    else: 
        avg_all=[]
        for u in users:
            user_profile=input_array[u,:-1]
            idx = np.nonzero(user_profile)[0]  # indices of interacted items

            avg=ID_energy[idx].mean()
            avg_all.append(avg)
        return np.mean(avg_all)
    

    
ID_liveness=items_data.set_index('item_id')['liveness']

def liveness_profiles(input_array, users, ID_liveness):
    
    if not users:
        return 0  
    else: 
        avg_all=[]
        for u in users:
            user_profile=input_array[u,:-1]
            idx = np.nonzero(user_profile)[0]  # indices of interacted items

            avg=ID_liveness[idx].mean()
            avg_all.append(avg)
        return np.mean(avg_all)
    

ID_dance=items_data.set_index('item_id')['danceability']
ID_instrument=items_data.set_index('item_id')['instrumentalness']

def dance_profiles(input_array, users, ID_dance):
    
    if not users:

        return 0
    
    else: 
        avg_all=[]
        for u in users:
            user_profile=input_array[u,:-1]
            idx = np.nonzero(user_profile)[0]  # indices of interacted items

            avg=ID_dance[idx].mean()
            avg_all.append(avg)
        return np.mean(avg_all)
    
def instrument_profiles(input_array, users, ID_instrument):
    
    if not users:
        return 0

    else: 
        avg_all=[]
        for u in users:
            user_profile=input_array[u,:-1]
            idx = np.nonzero(user_profile)[0]  # indices of interacted items

            avg=ID_instrument[idx].mean()
            avg_all.append(avg)
        return np.mean(avg_all)


## Exposure regression

In [ ]:

df=pd.DataFrame()


num_rows=num_items
idx= torch.randperm(num_rows)[:1000]
#samples=topk_itms[idx]
count_pos=0
count_neg=0


over_expos_itms = np.array(list(item2users.keys()))  # <-- convert set to list first
under_expos_itms = list(set(items_data['item_id'].unique()) - set(over_expos_itms)) # -1 --. MovieIDs in Ratings_df starts from 1

# Sample safely
sample_size =  1000
sampled_items = np.random.choice(under_expos_itms, size=sample_size, replace=False)

# Concatenate 1D arrays
all_items = np.concatenate([sampled_items, over_expos_itms])
rows_list = []  # Use a list for speed
for i in all_items:
        try:
                i=int(i)
                item_users=item2users.get(i,[])
                itm_genre=items_data.loc[items_data['item_id']==i,'genre' ].str.split('|').iloc[0]
                row = {
                        
                        'TargID': i,
                        #'playcounts': playcounts.get(i, 0),
                        #'profile_playcount': profile_playcount(all_array, item_users),
                        #'Artist Popularity': pop_artist[item2artist[i]],
                        #'Artis_prof_pop': Artist_profilr_pop(all_array, item_users),
                        #'energy_level_profile': energy_profiles(all_array, item_users, ID_energy),
                        #'liveness_profile': liveness_profiles(all_array, item_users, ID_liveness),
                        #'liveness':ID_liveness[i],
                        #'profiles_dance': dance_profiles(all_array, item_users, ID_dance),
                        #'target_dance': ID_dance[i],
                        #'instrument_profile': instrument_profiles(all_array, item_users, ID_instrument),
                        #'Profiles Time':recency_profiles(all_array,item_users, ID_Year),
                        #'Profiles Popularity': profile_popularity(all_array, item_users),

                         ## New features
                        'POP':pop_dict[i],
                        'SZ':profile_size(all_array, item_users, users_size),
                        'YR':ID_Year[i],
                        'GD':genre(all_array, ID2genre_hot, item_users, itm_genre ),

                        'ND': Targ_in_neighbs_profile (all_array, df_sim_users, item_users, i, drop_last_col=True),
                        'RI': neighbors_item_profile(all_array, sim_itms, item_users, i, drop_last_col=True),

                        #'AE':ID_energy[i],
                       # 'INS':ID_instrument[i],

                        

                        'Exposure': count[i]
                                    }
                rows_list.append(row)
        except Exception as e:
                print(f"Skipping item {i} due to error: {e}")
        continue

#df = pd.concat([df, pd.DataFrame([row])], ignore_index=True)
df = pd.DataFrame(rows_list)
        #indx=(torch.randperm(49)+1)[0:5]

   
        


In [ ]:
df['Exposure_']=np.log1p(df['Exposure'])


## Fitting the model

In [ ]:
scaler=MinMaxScaler()
array_scaled=scaler.fit_transform(df)

#df_scaled=pd.DataFrame(array_scaled, columns=df.columns)
df_scaled=pd.DataFrame(df, columns=df.columns)
#df_scaled['exposure_bin'] = pd.cut(df_scaled['Exposure'], bins=[0.0, 0.1, 0.3, 1.0], labels=['Low', 'Mid', 'High'],include_lowest=True)

X=df_scaled.drop(columns=['Exposure','TargID', 'Exposure_'])
y=df_scaled['Exposure_']


In [ ]:
df_results.to_csv(f"feature_importance_results_{recommender_name}_{data_name}.csv", index=False)

In [ ]:
df_results.to_csv(f"{data_name}_{recommender_name}.csv")

In [ ]:
plt.plot(df_results['feature_added'].values, df_results['R2'].values)

In [ ]:
X_test['B1'].hist()

In [ ]:
g1=X_test[X_test['B1']<0.1]
y_g1=y_test[X_test['B1']<0.1]

In [ ]:
y_hat_log = regr.predict(g1)
log_r2 = r2_score(y_g1, y_hat_log)
log_r2

In [ ]:


def asymmetric_objective(y_true, y_pred):
    # Calculate the residual
    residual = y_true - y_pred
    
    # Define your penalty ratio (e.g., 5.0 means under-predicting is 5x worse)
    penalty_ratio = 5
    
    # Gradient: If residual > 0 (under-predicted), multiply by penalty
    grad = np.where(residual > 0, -2 * penalty_ratio * residual, -2 * residual)
    
    # Hessian: Second derivative (constant for squared error)
    hess = np.where(residual > 0, 2 * penalty_ratio, 2.0)
    
    return grad, hess

# Initialize the model with the custom objective
regr = xgb.XGBRegressor(
    n_estimators=1000,
    learning_rate=0.01,
    max_depth=7,
    random_state=44
)


X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=44, shuffle=True)

# Apply the custom objective during fitting
# 2. Fit with Weights (XGBoost handles these very effectively)
regr.fit(X_train, y_train )

# 3. Predct and Evaluate
y_hat_log = regr.predict(X_test)
log_r2 = r2_score(y_test, y_hat_log)
print(f"XGBoost Log-space R2 Full Model: {log_r2}")
backward_selection_df=pd.DataFrame([{'feature': 'full', 'R2':log_r2 }])

X_filt=X.copy()
feat_names=['POP', 'ND', 'SZ', 'RI', 'YR', 'GD', 'AE', 'INS']
for i in feat_names[::-1]:

    X_filt=X_filt.drop(columns=[i])
    X_train, X_test, y_train, y_test = train_test_split(
        X_filt, y, test_size=0.2, random_state=44, shuffle=True)

    # Apply the custom objective during fitting
    # 2. Fit with Weights (XGBoost handles these very effectively)
    regr.fit(X_train, y_train )

    # 3. Predict and Evaluate
    y_hat_log = regr.predict(X_test)
    log_r2 = r2_score(y_test, y_hat_log)
    backward_selection_df = pd.concat([
    backward_selection_df, 
    pd.DataFrame([{'feature': i, 'R2': log_r2}])
], ignore_index=True)


    print(f"XGBoost Log-space R2 Dropping {i}: {log_r2}")
    MSE=mean_squared_error(y_test,y_hat_log )
    print(f"XGBoost MSE:{MSE}")
    # 4. Transform back for the Residual Plot
    y_hat_original = np.expm1(y_hat_log) 
    y_test_original = np.expm1(y_test)



residuals = y_test_original - y_hat_original

#6. Plotting Residuals in Original Space
plt.figure(figsize=(10, 6))
plt.scatter(y_hat_original, residuals, alpha=0.5)
plt.axhline(0, color='r', linestyle='--')
plt.xlabel('Predicted Exposure (Original Scale)')
plt.ylabel('Residuals (Error)')
#plt.title(f'Model name: {recommender_name}')
plt.show()

In [ ]:
import numpy as np
import xgboost as xgb
from sklearn.metrics import r2_score
from sklearn.metrics import mean_squared_error

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=44, shuffle=True)

def asymmetric_objective(y_true, y_pred):
    # Calculate the residual
    residual = y_true - y_pred
    
    # Define your penalty ratio (e.g., 5.0 means under-predicting is 5x worse)
    penalty_ratio = 5
    
    # Gradient: If residual > 0 (under-predicted), multiply by penalty
    grad = np.where(residual > 0, -2 * penalty_ratio * residual, -2 * residual)
    
    # Hessian: Second derivative (constant for squared error)
    hess = np.where(residual > 0, 2 * penalty_ratio, 2.0)
    
    return grad, hess

# Initialize the model with the custom objective
regr = xgb.XGBRegressor(
    #objective=asymmetric_objective, # Pass your custom objective here

    n_estimators=1000,
    learning_rate=0.01,
    max_depth=7,
    random_state=44
)

# Apply the custom objective during fitting
# 2. Fit with Weights (XGBoost handles these very effectively)
regr.fit(X_train, y_train )

# 3. Predict and Evaluate
y_hat_log = regr.predict(X_test)
log_r2 = r2_score(y_test, y_hat_log)
print(f"XGBoost Log-space R2: {log_r2}")
MSE=mean_squared_error(y_test,y_hat_log )
print(f"XGBoost MSE:{MSE}")
# 4. Transform back for the Residual Plot
y_hat_original = np.expm1(y_hat_log) 
y_test_original = np.expm1(y_test)



residuals = y_test_original - y_hat_original

 #6. Plotting Residuals in Original Space
plt.figure(figsize=(10, 6))
plt.scatter(y_hat_original, residuals, alpha=0.5)
plt.axhline(0, color='r', linestyle='--')
plt.xlabel('Predicted Exposure (Original Scale)')
plt.ylabel('Residuals (Error)')
#plt.title(f'Model name: {recommender_name}')
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score


# 1. Transform BOTH targets to Log-space for training/testing
# This makes the target distribution more Normal for the RF


# 2. Use your "Improved Configuration" - it's much better for correlated features
regr = RandomForestRegressor( max_depth=20, random_state=44)

# Create weights where higher actual exposure = more importance
weights = np.log1p(y_train) +1

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=44, shuffle=True)



regr.fit(X_train, y_train, sample_weight=weights)

# 3. Predict in Log-space
y_hat_log = regr.predict(X_test)

# 4. Calculate R2 in Log-space (Standard for surrogate models)
log_r2 = regr.score(X_test, y_test)
print(f"Log-space R2 score: {log_r2}")

# 5. Convert back to Original-space for Real-world Residuals
y_hat_original = np.expm1(y_hat_log)

# Note: Use the original y_test here, NOT the log version
residuals_original = y_hat_original - np.expm1(y_test)

# 6. Plotting Residuals in Original Space
plt.figure(figsize=(10, 6))
plt.scatter(y_hat_original, residuals_original, alpha=0.5)
plt.axhline(0, color='r', linestyle='--')
plt.xlabel('Predicted Exposure (Original Scale)')
plt.ylabel('Residuals (Error)')
plt.title(f'Model name: {recommender_name}')
plt.show()

In [ ]:
list(zip(regr.feature_names_in_, regr.feature_importances_))

In [ ]:
import xgboost as xgb
from sklearn.metrics import r2_score

# 1. Initialize the Regressor
# We use 'dart' or 'gbtree' booster for better regression
regr = xgb.XGBRegressor(
    objective='reg:squarederror',
    #quantile_alpha=0.8,
    n_estimators=1000, 
    max_depth=10,           # Deeper than RF to capture VAE complexity
    learning_rate=0.01,    # Small steps to avoid overfitting
    subsample=0.8,         # Use 80% of data per tree for robustness
    colsample_bytree=0.8,  # Similar to max_features='sqrt'
    random_state=44,
    n_jobs=-1
)

# 2. Fit with Weights (XGBoost handles these very effectively)
weights = np.log1p(y_train) +1
regr.fit(X_train, y_train, sample_weight=weights )

# 3. Predict and Evaluate
y_hat_log = regr.predict(X_test)
log_r2 = r2_score(y_test, y_hat_log)
print(f"XGBoost Log-space R2: {log_r2}")

# 4. Transform back for the Residual Plot
y_hat_original = np.expm1(y_hat_log)
y_test_original = np.expm1(y_test)



residuals = y_test_original - y_hat_original

 #6. Plotting Residuals in Original Space
plt.figure(figsize=(10, 6))
plt.scatter(y_hat_original, residuals, alpha=0.5)
plt.axhline(0, color='r', linestyle='--')
plt.xlabel('Predicted Exposure (Original Scale)')
plt.ylabel('Residuals (Error)')
#plt.title(f'Model name: {recommender_name}')
plt.show()

In [ ]:
list(zip(regr.feature_names_in_, regr.feature_importances_))

In [ ]:
import xgboost as xgb
from sklearn.metrics import r2_score
import numpy as np
import matplotlib.pyplot as plt

# ── Step 1: Log-transform exposure targets ──────────────────────────────────
# Exposure is heavy-tailed; log1p stabilizes variance and improves model fit.
# R² is evaluated in the original space after inverse transformation.
y_train_log = y_train
y_test_log  = y_test

# ── Step 2: Initialize XGBoost Regressor ───────────────────────────────────
# objective : reg:squarederror is consistent with R² evaluation metric.
# max_depth : 7 is consistent with what is reported in the paper and avoids
#             the severe overfitting risk of max_depth=200.
# learning_rate: 0.01 as reported in the paper (was 0.05 in old code).
# No sample_weight: removed to avoid amplifying high-exposure items and
#                   introducing bias into feature importance scores.
regr = xgb.XGBRegressor(
    objective='reg:squarederror',
    n_estimators=1000,
    max_depth=7,
    learning_rate=0.01,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=44,
    n_jobs=-1
)

# ── Step 3: Fit on log-transformed targets ──────────────────────────────────
# No sample weights — all items treated equally to avoid bias toward
# high-exposure items in the feature importance scores.
regr.fit(X_train, y_train_log)

# ── Step 4: Predict and evaluate ────────────────────────────────────────────
# Predict in log space, then inverse-transform to original exposure space.
y_pred_log      = regr.predict(X_test)
y_pred_original = np.expm1(y_pred_log)
y_test_original = np.expm1(y_test_log)

r2  = r2_score(y_test_original, y_pred_original)
mse = np.mean((y_test_original - y_pred_original) ** 2)

print(f"R²  (original space): {r2:.4f}")
print(f"MSE (original space): {mse:.4f}")

# ── Step 5: Feature importance (gain-based) ─────────────────────────────────
# feature_importances_ returns gain-based importance by default in XGBoost.
# Gain measures the average reduction in loss when a feature is used to split.
# These are the β values reported in Table 2 of the paper.
importances = list(zip(regr.feature_names_in_, regr.feature_importances_))
importances_sorted = sorted(importances, key=lambda x: x[1], reverse=True)

print("\nGain-based Feature Importances (β values in Table 2):")
print("-" * 40)
for feat, imp in importances_sorted:
    print(f"  {feat:<6}: {imp:.4f}")

# ── Step 6: Normalize importances to sum to 1 ───────────────────────────────
# This makes the β values in Table 2 directly comparable across models
# and interpretable as relative contributions.
total = sum(imp for _, imp in importances_sorted)
importances_normalized = [(feat, imp / total) for feat, imp in importances_sorted]

print("\nNormalized Feature Importances (sum to 1.0):")
print("-" * 40)
for feat, imp in importances_normalized:
    print(f"  {feat:<6}: {imp:.4f}")

# ── Step 7: Residual plot in original space ─────────────────────────────────
residuals = y_test_original - y_pred_original

plt.figure(figsize=(10, 6))
plt.scatter(y_pred_original, residuals, alpha=0.5)
plt.axhline(0, color='r', linestyle='--', linewidth=0.8)
plt.xlabel('Predicted Exposure (Original Scale)')
plt.ylabel('Residuals (Original Scale)')
plt.tight_layout()
plt.show()

# ── Step 8: Overfitting check ───────────────────────────────────────────────
# Compare train vs test R² to verify the model is not overfitting.
y_train_pred_log      = regr.predict(X_train)
y_train_pred_original = np.expm1(y_train_pred_log)
y_train_original      = np.expm1(y_train_log)

r2_train = r2_score(y_train_original, y_train_pred_original)
r2_test  = r2_score(y_test_original,  y_pred_original)

print(f"\nOverfitting check:")
print(f"  Train R²: {r2_train:.4f}")
print(f"  Test  R²: {r2_test:.4f}")
print(f"  Gap:      {r2_train - r2_test:.4f}  (should be < 0.05 ideally)")

# ── Step 9: What to report in the paper ────────────────────────────────────
# Add this sentence to Section 3.1:
# "Feature importance scores (β) reported in Table 2 correspond to the
#  gain-based importance metric provided by XGBoost, which measures the
#  average reduction in the squared error loss attributable to each feature
#  across all tree splits, normalized to sum to 1.0 across all features.
#  The surrogate is trained on log-transformed exposure values to handle
#  the heavy-tailed exposure distribution, and R² is evaluated in the
#  original exposure space after inverse transformation."

In [ ]:
import lightgbm as lgb
from sklearn.metrics import r2_score

# 1. Initialize the LightGBM Regressor
regr = lgb.LGBMRegressor(
    n_estimators=2000,
    learning_rate=0.01,
    num_leaves=64,          # Key parameter: higher = more complex logic
    max_depth=-1,           # Let it grow deep for your VAE outliers
    boosting_type='gbdt',   # Standard Gradient Boosting
    random_state=44,
    n_jobs=-1,
    importance_type='gain'  # Better for explaining VAE feature importance
)

# 2. Fit the model (using your log-transformed y_train)
# LightGBM handles sample weights extremely well
weights = np.log1p(y_train) + 1
regr.fit(X_train, y_train, sample_weight=weights)

# 3. Predict and evaluate in Log-space
y_hat_log = regr.predict(X_test)
log_r2 = r2_score(y_test, y_hat_log)
print(f"LightGBM Log-space R2: {log_r2}")

# 4. Transform back for the Residual Plot
y_hat_original = np.expm1(y_hat_log)
y_test_original = np.expm1(y_test)
residuals = y_hat_original - y_test_original

 #6. Plotting Residuals in Original Space
plt.figure(figsize=(10, 6))
plt.scatter(y_hat_original, residuals, alpha=0.5)
plt.axhline(0, color='r', linestyle='--')
plt.xlabel('Predicted Exposure (Original Scale)')
plt.ylabel('Residuals (Error)')
#plt.title(f'Model name: {recommender_name}')
plt.show()

In [ ]:
list(zip(regr.feature_names_in_, regr.feature_importances_))

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd
df_scaled['y_pred']=model.predict(X)
# 1. Create 10-20 "buckets" for your continuous variables
# This groups values like 0.027 and 0.035 into one 'bin'
df_scaled['B4_bins'] = pd.cut(df_scaled['Neighbors Recommendations'], bins=5)
df_scaled['B1_bins'] = pd.cut(df_scaled['Genre Affinity'], bins=5)

# 2. Use the bins for your pivot table and calculate the MEAN exposure
heatmap_data = df_scaled.pivot_table(index="B1_bins", 
                            columns='B4_bins', 
                            values="y_pred", 
                            aggfunc='mean')

# 3. Plotting
plt.figure(figsize=(10, 5))
sns.heatmap(heatmap_data, 
            cmap='bwr', 
            vmin=0, 
            vmax=1, 
            center=0.5,
            cbar_kws={'label': 'Exposure'})

# Formatting
plt.xlabel('Neighbors Recommendations')
plt.ylabel('Genre Affinity')
plt.title('Influence Heatmap')
plt.gca().invert_yaxis()

plt.show()

## Scatterplot

In [ ]:
import matplotlib.pyplot as plt

features=X.columns.values

n_cols = 3
n_rows = (len(features) + n_cols - 1) // n_cols

fig, axes = plt.subplots(n_rows, n_cols, figsize=(15, 5))
axes = axes.flatten()

for ax, feat in zip(axes, features):
    df_scaled.plot.scatter(x=feat, y='Exposure', ax=ax)
    ax.set_title(f' Exposure vs {feat}')

# Remove empty plots
for i in range(len(features), len(axes)):
    fig.delaxes(axes[i])

plt.tight_layout()

plt.show()



## Correlations

In [ ]:
df.corr()['Exposure'].sort_values(ascending=False)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=34, shuffle=True
)

## Random Forest Regressor

In [ ]:
from sklearn.ensemble import RandomForestRegressor

regr = RandomForestRegressor(max_depth=8, random_state=44)
regr.fit(X_train, y_train)
print(f"The R2 score is {regr.score(X_test, y_test)}")
print(f"Features Importance {dict(zip(X.columns, regr.feature_importances_))}")
y_hat=regr.predict(X_test)

residuals=y_hat-y_test
plt.scatter(y_hat,residuals )
plt.axhline(0, color='r', linestyle='--')
plt.xlabel('y_pred')
plt.ylabel('Residuals')
plt.title(f"recommender model: {recommender_name}")
plt.show()


In [ ]:
from sklearn.metrics import  mean_squared_error
mean_squared_error(y_test, y_hat)

In [ ]:
dict(zip(X.columns, regr.feature_importances_))

In [ ]:
df.corr()['Exposure'].sort_values(ascending=False)

## Residuals plot

In [ ]:
y_hat=regr.predict(X_test)
residuals=y_hat-y_test
plt.scatter(y_hat,residuals )
plt.axhline(0, color='r', linestyle='--')
plt.xlabel('y_pred')
plt.ylabel('Residuals')
plt.title(f"recommender model: {recommender_name}")
plt.show()

In [ ]:

df_long = all_df_data.melt(id_vars='user_ids', var_name='item_id', value_name='rating')
# 2. Keep only the rows where the rating is 1 (non-zero)
df_final = df_long[df_long['rating'] > 0].reset_index(drop=True)

# 3. Ensure your columns are correctly named for your next step
df_final['timestamp:float']=df_final["rating"]


In [ ]:
df_final=df_final.rename(columns={"user_ids":"user_id:token", "item_id": "item_id:token", "rating": "playcount:float" })

In [ ]:
df_final.to_csv('my_dataset_LASTFM.inter', sep='\t', index=False)

In [ ]:
from sklearn.feature_selection import SequentialFeatureSelector
from sklearn.metrics import r2_score
import xgboost as xgb
import numpy as np

# Define model
model = xgb.XGBRegressor(
    n_estimators=1000,
    max_depth=7,
    learning_rate=0.001,
    random_state=42
)

# Backward selection
sfs = SequentialFeatureSelector(
    estimator=model,
    n_features_to_select='auto',  # automatically finds best number
    direction='backward',          # this is the key parameter
    scoring='r2',
    cv=5                           # cross validation folds
)

sfs.fit(X_train, y_train)

# Get selected features
selected_features = X_train.columns[sfs.get_support()]
print("Selected features:", selected_features)